# Ask AOPWiki a question

In [1]:
question = "Which Adverse Outcome Pathways represent thyroid issues in humans or mammals, and which genes are described to be related to these pathways? What method was used to annotate these gene relationships?"

In [2]:
import os
import sys
from dotenv import load_dotenv
from mcp import Client as MCPClient, StdioServerParameters
from pydantic_ai.usage import UsageLimits
from rdfsolve.pydantic_ai import research_agent, save_answer
from rdfsolve.mcp import query_answer, read_plan
import pandas as pd
from IPython.display import Markdown, display

load_dotenv("../.env");

## Setup

In [3]:
server = MCPClient(
    StdioServerParameters(
        command=sys.executable,
        args=["-m", "rdfsolve.mcp", "--schema", "../data/aopwikirdf.schema.json",
              "--source-id", "aopwikirdf", "--log", "output/investigation-session.json"],
    ),
    read_timeout_seconds=900,
)

## Question

In [4]:
async with server:
    agent = await research_agent(
        server,
        os.getenv("OPENAI_MODEL", "openai:gpt-5.4-mini-2026-03-17"),
        model_settings={"max_tokens": 2000},
        retries=2,
    )
    answer = await agent.run(
        question,
        usage_limits=UsageLimits(request_limit=20, tool_calls_limit=50, total_tokens_limit=60000),
    )
    display(Markdown(answer.output.text))
    usage = answer.usage
    references = [item.reference for item in answer.output.results]
    result = await query_answer(server, references, name="Pathways, genes and annotation methods") if references else None
    investigation = await read_plan(server)
save_answer(answer, "output/investigation-session.json", question=question)
if result is None:
    raise ValueError("The agent did not produce an answer query. Its explanation and attempted plan are saved in output/investigation-session.json.")

I found thyroid-related Adverse Outcome Pathways that explicitly mention human or mammalian relevance. The strongest direct matches are:
- AOP 110: Inhibition of iodide pump activity leading to follicular cell adenomas and carcinomas (in rat and mouse)
- AOP 119: Inhibition of thyroid peroxidase leading to follicular cell adenomas and carcinomas (in rat and mouse)
- AOP 128: Kidney dysfunction by decreased thyroid hormone
- AOP 134: Sodium Iodide Symporter (NIS) Inhibition and Subsequent Adverse Neurodevelopmental Outcomes in Mammals
- AOP 152: Interference with thyroid serum binding protein transthyretin and subsequent adverse human neurodevelopmental toxicity
- AOP 162: Enhanced hepatic clearance of thyroid hormones leading to thyroid follicular cell adenomas and carcinomas in the rat and mouse
- AOP 175: Thyroperoxidase inhibition leading to altered amphibian metamorphosis
- AOP 176: Sodium Iodide Symporter (NIS) Inhibition leading to altered amphibian metamorphosis
- AOP 188: Iodotyrosine deiodinase (IYD) inhibition leading to altered amphibian metamorphosis
- AOP 189: Type I iodothyronine deiodinase (DIO1) inhibition leading to altered amphibian metamorphosis
- AOP 190: Type II iodothyronine deiodinase (DIO2) inhibition leading to altered amphibian metamorphosis

For gene relationships, the retrieved route shows genes linked through Key Event Relationships. Example gene identifiers observed in the thyroid pathway route include TSHZ1, SLC5A5, and TG. The full result set contains many more gene identifiers, but the preview only exposed these examples.

Method used to annotate gene relationships: geneDetectedByNER, labeled as "gene detected by BERN2 NER+EL (featured recall-extending method)". The alternate route geneDetectedByRegex is described as "gene detected by HGNC dictionary regex (baseline method)".

One caution: the database result set is large, and the preview only shows a sample of the gene list. The answer table contains 1312 rows and 27 matching thyroid-related AOPs in the filtered result set.

## Strategy

In [5]:
print(investigation["plan"]["selection"])
display(Markdown("### Requested classes"))
display(pd.DataFrame(investigation["plan"]["columns"]).reindex(columns=["class", "role"]))
display(Markdown("### Required conditions — all must match"))
display(pd.DataFrame(investigation["plan"]["where"]))
display(Markdown("### Routes considered"))
display(pd.DataFrame(investigation["routes"]).reindex(columns=["route"]))
display(Markdown("### Searches executed"))
display(pd.DataFrame(investigation["searches"]).reindex(columns=["terms", "fields"]))

thyroid-related AOPs in humans or mammals with genes and annotation method


### Requested classes

,class,role
0,Adverse Outcome Pathway,source
1,Gene identifier,target


### Required conditions — all must match

,kind,fields,terms,iris
0,http://aopkb.org/aop_ontology#AdverseOutcomePa...,"[description, abstract, label]",[thyroid],[]
1,http://aopkb.org/aop_ontology#AdverseOutcomePa...,"[description, abstract, label]","[human, mammal, mammalian]",[]


### Routes considered

,route
0,Adverse Outcome Pathway → Key Event Relationsh...
1,Adverse Outcome Pathway → Key Event → Gene ide...
2,Adverse Outcome Pathway → Key Event Relationsh...
3,Adverse Outcome Pathway → Key Event → Gene ide...
4,Adverse Outcome Pathway → Key Event Relationsh...
5,Adverse Outcome Pathway → Key Event → Gene ide...


### Searches executed

,terms,fields
0,"[thyroid, human, mammal, gene relationship, an...",NaN
1,"[thyroid, human, mammal]",NaN
2,[thyroid],NaN
3,"[BERN2, HGNC dictionary regex, geneDetectedByN...",NaN
4,[thyroid],NaN


In [6]:
show_table = True
if show_table:
    display(Markdown(f"## {result.name}"))
    print(result.coverage["basis"])
    if result.coverage["status"] == "partial":
        print("This is a partial result; a query or its source selection reached a limit.")
    with pd.option_context("display.max_columns", None):
        display(result.table())

## Answer

Selected routes and explicit filters, not exhaustive topic coverage


,Adverse Outcome Pathway · IRI,Adverse Outcome Pathway · label,Adverse Outcome Pathway — has key event relationship → Key Event Relationship · IRI,Adverse Outcome Pathway — has key event relationship → Key Event Relationship · description,Adverse Outcome Pathway — has key event relationship → Key Event Relationship · label,Key Event Relationship — gene detected by BERN2 NER+EL (featured recall-extending method) → Gene identifier · IRI,Key Event Relationship — gene detected by BERN2 NER+EL (featured recall-extending method) → Gene identifier · label
0,https://identifiers.org/aop/110,AOP 110,https://identifiers.org/aop.relationships/305,"Thyroid hormones (THs), thyroxine (T4) and tri...",KER 305,https://identifiers.org/hgnc/10669,TSHZ1
1,https://identifiers.org/aop/110,AOP 110,https://identifiers.org/aop.relationships/305,"Thyroid hormones (THs), thyroxine (T4) and tri...",KER 305,https://identifiers.org/hgnc/11040,SLC5A5
2,https://identifiers.org/aop/110,AOP 110,https://identifiers.org/aop.relationships/305,"Thyroid hormones (THs), thyroxine (T4) and tri...",KER 305,https://identifiers.org/hgnc/11764,TG
3,https://identifiers.org/aop/110,AOP 110,https://identifiers.org/aop.relationships/305,"Thyroid hormones (THs), thyroxine (T4) and tri...",KER 305,https://identifiers.org/hgnc/11782,TH
4,https://identifiers.org/aop/110,AOP 110,https://identifiers.org/aop.relationships/305,"Thyroid hormones (THs), thyroxine (T4) and tri...",KER 305,https://identifiers.org/hgnc/12015,TPO
...,...,...,...,...,...,...,...
1307,https://identifiers.org/aop/7,AOP 7,https://identifiers.org/aop.relationships/394,The ovarian cycle irregularities impact on rep...,KER 394,https://identifiers.org/hgnc/9081,PLOD1
1308,https://identifiers.org/aop/7,AOP 7,https://identifiers.org/aop.relationships/394,The ovarian cycle irregularities impact on rep...,KER 394,https://identifiers.org/hgnc/9445,PRL
1309,https://identifiers.org/aop/7,AOP 7,https://identifiers.org/aop.relationships/396,Aromatase is the cytochrome P450 enzyme comple...,KER 396,https://identifiers.org/hgnc/2594,CYP19A1
1310,https://identifiers.org/aop/7,AOP 7,https://identifiers.org/aop.relationships/396,Aromatase is the cytochrome P450 enzyme comple...,KER 396,https://identifiers.org/hgnc/9208,POR


## Links in the answer


In [12]:
if result.bindings:
    display(Markdown(result.diagram(instances=True, row=7)))

Observed routes retained in this result; not sentence-level citations.

```mermaid
flowchart LR
subgraph R0["1 observed match#40;es#41;#59; queries: 23#59; graph: http://aopwiki.org/"]
N0_0["AOP 119<br/>https://identifiers.org/aop/119<br/>Adverse Outcome Pathway"]
N0_1["KER 305<br/>https://identifiers.org/aop.relationships/305<br/>Key Event Relationship"]
N0_2["SLC5A5<br/>https://identifiers.org/hgnc/11040<br/>Gene identifier"]
N0_0 -->|"http://aopkb.org/aop_ontology#35;has_key_event_relationship"| N0_1
N0_1 -->|"https://aopwiki.rdf.bigcat-bioinformatics.org/geneDetectedByNER"| N0_2
end
```

Each row shows a returned connection. The package reads names and other requested fields separately; several values stay together in one cell. Empty fields have no returned value.

Annotation names and definitions come from the saved schema. They describe the predicates used; they do not establish experimental evidence.

## Reuse the answer

The connection query records the selected routes and filters. Follow-up queries retrieve the fields shown in the table. Pages and retries run inside rdfsolve.

Optionally save the connection and field queries with their SHACL paths, or the returned data as RDF. The RDF keeps intermediate records and the original predicates.

In [8]:
from pathlib import Path
from rdflib import Graph

show_query = True
save_shacl = True
save_subset = True

if show_query:
    print(result.query)
if save_shacl or save_subset:
    Path("output").mkdir(exist_ok=True)
if save_shacl:
    result.to_shacl().serialize("output/answer.shacl.ttl", format="turtle")
if save_subset:
    subset = Graph()
    for record in result.records():
        subset += record.to_graph()
    subset.serialize("output/answer.ttl", format="turtle")

SELECT DISTINCT ?_route ?source ?via1 ?target ?_graph WHERE {
{ VALUES ?_graph { <http://aopwiki.org/> } GRAPH ?_graph { VALUES (?_route ?link1 ?link2) { (0 <http://aopkb.org/aop_ontology#has_key_event_relationship> <https://aopwiki.rdf.bigcat-bioinformatics.org/geneDetectedByNER>) }
?source a <http://aopkb.org/aop_ontology#AdverseOutcomePathway> . ?source ?link1 ?via1 . ?via1 a <http://aopkb.org/aop_ontology#KeyEventRelationship> . ?via1 ?link2 ?target . ?target a <http://edamontology.org/data_1025> . FILTER(!sameTerm(?via1, ?source)) FILTER(!sameTerm(?target, ?source)) FILTER(!sameTerm(?target, ?via1)) FILTER(EXISTS { ?source (<http://purl.org/dc/elements/1.1/description>|<http://purl.org/dc/terms/abstract>|<http://www.w3.org/2000/01/rdf-schema#label>) ?match0 . FILTER(CONTAINS(LCASE(STR(?match0)), LCASE("thyroid"))) }) FILTER(EXISTS { ?source (<http://purl.org/dc/elements/1.1/description>|<http://purl.org/dc/terms/abstract>|<http://www.w3.org/2000/01/rdf-schema#label>) ?match1 . FIL

## See what it used

The table lists the tools used. Open an entry below it to see its arguments and answer, or the SPARQL queries and returned rows.

In [9]:
from rdfsolve.query_log import QueryLog

log = QueryLog.read("output/investigation-session.json")
display(Markdown("### Tool calls"))
display(log.tools())
print(usage)

### Tool calls

,Tool,Name,Status,Queries
0,search,http://aopkb.org/aop_ontology#AdverseOutcomePa...,partial,"[1, 2, 3, 4, 5, 6]"
1,schema,http://aopkb.org/aop_ontology#AdverseOutcomePa...,complete,[]
2,schema,http://edamontology.org/data_1025,complete,[]
3,search,http://aopkb.org/aop_ontology#AdverseOutcomePa...,partial,"[7, 8, 9, 10, 11, 12]"
4,plan,Adverse Outcome Pathways representing thyroid ...,complete,[]
5,answer,Answer,failed,[]
6,answer,Answer,failed,[]
7,search,http://aopkb.org/aop_ontology#AdverseOutcomePa...,complete,"[13, 14, 15, 16, 17]"
8,answer,Answer,failed,[]
9,search,http://aopkb.org/aop_ontology#KeyEventRelation...,complete,[18]


RunUsage(details={'reasoning_tokens': 0}, requests=13, tool_calls=10)
